# 02. 거래 건수 수치화
> 계획서: `계획/P2_건수수치화.md`

원본의 건수는 **구간(0건, 1건, 2건, 3~5건, … 50건초과)** 으로 되어 있습니다. 이 노트북은 구간을 숫자로 바꾸는 방법을 비교하고, 이후 분석에서 쓸 방법을 정합니다.

| 표현 | 설명 | 장점 | 한계 |
|---|---|---|---|
| R1 순서 코드 | 0~9 | 가정이 가장 적음, 단계 변화로 해석 | 단계 간격이 같지 않음 |
| R2 근사 건수 | 구간 중앙값 (50건초과는 cap) | 합산 가능(총 활동량) | 50건초과를 얼마로 볼지 가정 필요 |
| R3 로그 근사 건수 | log(1 + 근사 건수) | 비율 변화에 가까움 | R2의 한계를 그대로 가짐 |
| R4 이용 여부 | 코드 > 0 | 가정 없음 | 정보 손실이 큼 |

In [ ]:
import os, sys, json, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ── 설정: 여기만 고치세요 ────────────────────────────────────────────
WORK_DIR = os.environ.get("CNT_WORK_DIR") or None   # 기본값: 레포의 data/work_cnt (.gitignore 대상)
# ────────────────────────────────────────────────────────────────────

def _find_lib():
    for base in [Path.cwd(), *Path.cwd().parents]:
        for cand in (base / "code", base / "analysis" / "code"):
            if (cand / "cntlib.py").exists():
                return cand
    raise SystemExit("cntlib.py 를 찾지 못했습니다. 노트북을 analysis/notebooks 에서 여세요.")

sys.path.insert(0, str(_find_lib()))
import cntlib as cl
cl.setup_korean_font()
WORK = cl.resolve_work_dir(WORK_DIR)
OUT = WORK / "output"
print("산출물 폴더:", WORK)

In [ ]:
panel = cl.CountPanel.load(WORK / "interim" / "count_panel.npz")
_m = np.load(WORK / "interim" / "sample_masks.npz")
masks = {k: _m[k] for k in _m.files}
if panel.synthetic:
    print(cl.SYNTHETIC_NOTICE)
pops = {"P36": masks["P36"], "P35+": masks["P35p"]}
print({k: int(v.sum()) for k, v in pops.items()}, "개 기업")

## 1. 구간과 대표 건수

In [ ]:
mids = [f"{m:g}" for m in cl.BAND_MIDS] + [f"cap (기본 {cl.DEFAULT_CAP:g}, 민감도 51·100)"]
cl.show_table(pd.DataFrame({"코드": range(10), "구간 라벨": panel.band_labels, "대표 건수": mids}), "구간 코드와 대표 건수", OUT / "nb02", "bands")

## 2. 변수별 이용률·천장 (P36 vs P35+)
- 이용률 = 코드 > 0 인 칸 비율, 천장 = 코드 9(50건초과) 비율. **천장이 높은 변수**는 건수가 줄어도 코드가 안 바뀌어 변화를 놓칩니다.

In [ ]:
rows = []
for pname, m in pops.items():
    for c in cl.CNT_COLS:
        v = panel.cnt[c][m][panel.observed[m]]
        v = v[v >= 0]
        rows.append({"표본": pname, "변수": cl.SHORT[c], "관측 칸": cl.sup(len(v)), "이용 비율(%)": round((v > 0).mean() * 100, 1),
                     "0건 비율(%)": round((v == 0).mean() * 100, 1), "천장(50건초과) 비율(%)": round((v == 9).mean() * 100, 1),
                     "코드 중앙값": float(np.median(v))})
usage = pd.DataFrame(rows)
cl.show_table(usage, "변수별 이용률·천장", OUT / "nb02", "usage_ceiling")
fig, ax = plt.subplots(figsize=(8, 3.6))
u36 = usage[usage["표본"] == "P36"]
ax.bar(range(len(u36)), u36["천장(50건초과) 비율(%)"], color="#2a78d6")
ax.set_xticks(range(len(u36))); ax.set_xticklabels(u36["변수"], rotation=30)
ax.set_ylabel("천장 비율 (%)"); ax.set_title("P36: 최상위 구간(50건초과) 비율")
plt.tight_layout(); plt.show()

## 3. 합산 활동량 지수와 cap 민감도
국내 5채널(창구·인터넷·스마트·폰·ATM) 근사 건수 합 = **총 채널 활동량 I1**. 50건초과를 얼마로 볼지(cap) 바꿔 가며 I1이 얼마나 달라지는지 봅니다.

In [ ]:
P = masks["P36"]
Z = {c: cl.codes_float(panel.cnt[c][P], panel.observed[P]) for c in cl.CH5}
def I1(cap):
    return sum(cl.approx_count(Z[c], cap) for c in cl.CH5)
I51, I60, I100 = I1(51), I1(60), I1(100)
n_ceil = sum(np.where(np.isnan(Z[c]), 0, (Z[c] == 9)).astype(int) for c in cl.CH5)
ok = ~np.isnan(I60)
dist = pd.Series(n_ceil[ok]).value_counts(normalize=True).reindex([0, 1, 2], fill_value=0) * 100
# 5채널이 모두 0건이면 I1 = 0 이라 cap 과 무관하다 (변화 0). 분모 0 을 1e-9 로 막으면 100% 로 잘못 계산되므로 따로 처리한다.
i51, i100 = I51[ok], I100[ok]
rel = np.zeros(i51.shape)
pos = i51 > 0
rel[pos] = np.abs(i100[pos] / i51[pos] - 1)
has_ceil = n_ceil[ok] >= 1
ordsum = sum(np.where(np.isnan(Z[c]), 0, Z[c]) for c in cl.CH5)
rho = pd.Series(I60[ok]).rank().corr(pd.Series(ordsum[ok]).rank())   # 순위의 피어슨 상관 = 스피어먼 상관 (scipy 불필요)
tab = pd.DataFrame([
    ("국내 5채널 중 천장 채널이 0개인 기업-월(%)", round(dist[0], 1)),
    ("천장 채널이 1개인 기업-월(%)", round(dist[1], 1)),
    ("천장 채널이 2개 이상인 기업-월(%)", round(float(100 - dist[0] - dist[1]), 1)),
    ("|I1(cap100)/I1(cap51) - 1| 중앙값(%)", round(float(np.median(rel)) * 100, 2)),
    ("|I1(cap100)/I1(cap51) - 1| 90분위(%)", round(float(np.percentile(rel, 90)) * 100, 2)),
    ("|I1(cap100)/I1(cap51) - 1| 99분위(%)", round(float(np.percentile(rel, 99)) * 100, 2)),
    ("[천장 채널이 1개 이상인 기업-월만] |I1(cap100)/I1(cap51) - 1| 중앙값(%)", round(float(np.median(rel[has_ceil])) * 100, 2)),
    ("[천장 채널이 1개 이상인 기업-월만] |I1(cap100)/I1(cap51) - 1| 90분위(%)", round(float(np.percentile(rel[has_ceil], 90)) * 100, 2)),
    ("Spearman: I1(cap60) vs 순서 코드 합", round(float(rho), 4)),
], columns=["지표", "값"])
cl.show_table(tab, "I1의 cap 민감도 (P36)", OUT / "nb02", "i1_cap_sensitivity")

## 3b. 구간 하한 지수와 상한 시나리오 (구간 안 분포 가정에 대한 민감도)
구간 중앙값은 실제 건수가 아니라 *근사*입니다. 모든 채널이 구간의 **하한**에 있다고 본 합(I1_low)과 **상한 시나리오**(I1_high)를 구해, 합산 지수가 구간 안 분포 가정에 얼마나 민감한지 봅니다.
**I1_high 는 실제 상한이 아닙니다.** `50건초과`에는 상한이 없어 대표값 100으로 둔 *시나리오*일 뿐이고, 실제 건수는 100을 넘을 수 있습니다 (원건수가 없어 확인 불가).
그래서 `50건초과` 채널이 합산 지수에 얼마나 기여하는지도 같이 봅니다. 위축 탐지 판정이 이 지수들에서 유지되는지는 P4·P7에서 확인합니다.

In [ ]:
Il = sum(cl.bound_count(Z[c], "low") for c in cl.CH5)
Ih = sum(cl.bound_count(Z[c], "high", 100) for c in cl.CH5)
okb = ~np.isnan(Il) & (Il > 0)
rr = Ih[okb] / Il[okb]
def rk(a, b):
    return round(float(pd.Series(a[okb]).rank().corr(pd.Series(b[okb]).rank())), 4)
ceil_part = sum(np.where(Z[c] == 9, cl.DEFAULT_CAP, 0.0) for c in cl.CH5)      # 50건초과 채널이 I1(cap60)에 넣는 몫
cshare = ceil_part[okb] / I60[okb]
tab = pd.DataFrame([
    ("50건초과 채널이 I1(cap60)에 기여하는 비중 중앙값(%)", round(float(np.median(cshare)) * 100, 1)),
    ("50건초과 채널이 I1(cap60)에 기여하는 비중 90분위(%)", round(float(np.percentile(cshare, 90)) * 100, 1)),
    ("I1_high(상한 시나리오) / I1_low 중앙값", round(float(np.median(rr)), 2)),
    ("I1_high(상한 시나리오) / I1_low 90분위", round(float(np.percentile(rr, 90)), 2)),
    ("I1_high(상한 시나리오) / I1_low 99분위", round(float(np.percentile(rr, 99)), 2)),
    ("Spearman: I1_low vs I1(cap60)", rk(Il, I60)),
    ("Spearman: I1(cap60) vs I1_high", rk(I60, Ih)),
    ("Spearman: I1_low vs I1_high", rk(Il, Ih)),
], columns=["지표", "값"])
cl.show_table(tab, "구간 하한·상한 지수 비교 (P36)", OUT / "nb02", "i1_bounds")

## 4. 1년 전 같은 달 대비 코드 변화는 얼마나 자주 일어나나
채널 변화 이벤트(감소·증가)를 정의할 때 임계값의 근거가 됩니다. 변화가 대부분 0이면 ±1 코드도 의미 있는 이벤트입니다.

In [ ]:
rows = []
for c in cl.CNT_COLS:
    Zc = cl.codes_float(panel.cnt[c][P], panel.observed[P])
    d = (Zc[:, 12:] - Zc[:, :-12]).ravel()
    d = d[~np.isnan(d)]
    row = {"변수": cl.SHORT[c], "비교 칸": cl.sup(len(d))}
    for lab, cond in (("<= -3", d <= -3), ("-2", d == -2), ("-1", d == -1), ("0", d == 0), ("+1", d == 1), ("+2", d == 2), (">= +3", d >= 3)):
        row[f"{lab}(%)"] = round(float(cond.mean() * 100), 1)
    rows.append(row)
cl.show_table(pd.DataFrame(rows), "1년 전 같은 달 대비 코드 변화 분포 (P36)", OUT / "nb02", "yoy_code_change")

## 5. 결정 기록 (계획서 P2의 사전 규칙을 따른다)

In [ ]:
share_ceiling_max = float(usage[usage["표본"] == "P36"]["천장(50건초과) 비율(%)"].max())
spec = {
    "aggregate_activity": "I1 = 국내 5채널 근사 건수 합 (창구·인터넷·스마트·폰·ATM)",
    "approx_count": {"mids": cl.BAND_MIDS, "cap_primary": cl.DEFAULT_CAP, "cap_sensitivity": [51, 100]},
    "single_channel_change": "순서 코드 차이 (평균하지 않고 단계 변화로 본다)",
    "separate_axes": ["자동이체", "외환(수출·수입)"],
    "ceiling_max_pct_P36": share_ceiling_max,
    "cap_sensitivity_required": bool(share_ceiling_max >= 25),
    "bounds_sensitivity": "I1_low(구간 하한 합), I1_high(상한 시나리오: 50건초과=100, 실제 상한 아님) 민감도 필수 (Codex 검증 반영)",
}
cl.save_json(WORK / "interim" / "quant_spec.json", spec)
print(json.dumps(spec, ensure_ascii=False, indent=2))

## 체크포인트 (Claude에게 알려줄 것)
- [ ] 변수별 이용률·천장 표 (특히 천장이 25% 이상인 변수가 있는가)
- [ ] I1의 cap 민감도: 천장 채널 비율과 |I1(cap100)/I1(cap51)−1|의 90·99분위
- [ ] 50건초과 채널이 I1에 기여하는 비중(중앙값·90분위), I1_high(상한 시나리오)/I1_low의 중앙값·99분위와 세 지수 사이의 Spearman
- [ ] 코드 변화 분포에서 0이 아닌 비율 (이벤트 임계값 ±1 코드가 적절한가)
- [ ] 결정 기록의 `cap_sensitivity_required` 값

**다음**: `P2` 체크리스트 → `P3` 계획 확인 → `03_채널구조탐색.ipynb`.